# Análise de GHI e Resultados dos Modelos para as Localidades de Fábricas de Veículos Elétricos

## Apresentação

Este notebook apresenta a aplicação do pipeline de previsão diária de Irradiância Solar Global Horizontal (GHI) em dez localidades associadas a fábricas de veículos elétricos.

O estudo foi desenvolvido em três etapas principais. Inicialmente, foram obtidos dados solares por meio da API da National Solar Radiation Database (NSRDB). Em seguida, realizou-se a caracterização das séries temporais de GHI. Por fim, o pipeline de pré-processamento e aprendizado de máquina foi aplicado individualmente a cada localidade, utilizando os modelos XGBoost e MLP.

O período histórico analisado compreende os anos de 2019 a 2024, correspondentes aos anos efetivamente disponibilizados pelo produto GOES Aggregated PSM v4 no momento da coleta.

## 1. Configuração do ambiente

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "cadernos_jupyter":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

from codigo_fonte.localidades_ev import (
    LOCALIDADES_EV,
    dataframe_localidades,
    distancia_haversine_km,
)
from treinar_todas_localidades import (
    MANIFESTO_DADOS,
    OUTPUT_DIR,
    calcular_sha256,
    nome_arquivo,
    validar_csv_nrel_localidade,
)

RESULTADOS_DIR = PROJECT_ROOT / "resultados" / "todas_localidades"
PREVISOES_DIR = RESULTADOS_DIR / "previsoes"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 100)

print("Ambiente configurado para a análise das dez localidades.")

## 2. Aquisição e organização dos dados

### 2.1 Fonte dos dados solares

Os dados solares foram obtidos por meio da API oficial da **National Solar Radiation Database (NSRDB)**, atualmente disponibilizada pela National Laboratory of the Rockies (NLR). Utilizou-se o produto **GOES Aggregated PSM v4**, com resolução temporal horária de 60 minutos.

Para cada localidade, os valores horários de GHI foram agrupados por dia por meio da média aritmética. Dessa forma, o valor diário representa a irradiância média horária observada ao longo das 24 horas, expressa em `W/m²`.

A mesma metodologia de coleta foi aplicada às dez localidades, preservando-se o período de 1º de janeiro de 2019 a 31 de dezembro de 2024.

### 2.2 Localidades selecionadas e fontes das coordenadas

A identificação de cada fábrica foi confirmada por meio de página institucional da respectiva empresa. As coordenadas geográficas foram associadas ao centro do elemento industrial correspondente no OpenStreetMap, consultado por meio do serviço Nominatim.

A tabela a seguir apresenta as localidades, os endereços utilizados e as respectivas fontes de referência.

In [ ]:
fontes = dataframe_localidades()[
    [
        "nome", "pais", "endereco", "lat", "lon",
        "fonte_localidade", "fonte_coordenadas",
    ]
].copy()

fontes["Fonte institucional"] = fontes["fonte_localidade"].map(
    lambda url: f'<a href="{url}" target="_blank">Página oficial</a>'
)
fontes["Fonte geográfica"] = fontes["fonte_coordenadas"].map(
    lambda url: f'<a href="{url}" target="_blank">OpenStreetMap</a>'
)

tabela_fontes = fontes.drop(columns=["fonte_localidade", "fonte_coordenadas"]).rename(
    columns={
        "nome": "Localidade",
        "pais": "País",
        "endereco": "Endereço",
        "lat": "Latitude",
        "lon": "Longitude",
    }
)

display(HTML(tabela_fontes.to_html(index=False, escape=False)))

### 2.3 Caracterização da base de dados

Antes da aplicação dos modelos, foram verificados a cobertura temporal, a ausência de datas duplicadas, a unidade retornada pela API e a correspondência entre as coordenadas das fábricas e os pontos da grade espacial do NSRDB.

A distância apresentada na tabela corresponde à separação entre a coordenada de referência da fábrica e o centro da célula da grade utilizada pelo produto solar.

In [ ]:
if not MANIFESTO_DADOS.exists():
    raise FileNotFoundError("O manifesto dos dados NSRDB não foi localizado.")

manifesto = pd.read_csv(MANIFESTO_DADOS)
bases = {}
resumo_dados = []
datas_esperadas = pd.date_range("2019-01-01", "2024-12-31", freq="D")

for local in LOCALIDADES_EV:
    arquivo = OUTPUT_DIR / f"{nome_arquivo(local['nome'])}.csv"
    valido, motivo = validar_csv_nrel_localidade(arquivo, local)
    dados = pd.read_csv(arquivo, parse_dates=["data"])
    bases[local["nome"]] = dados

    registro_manifesto = manifesto.loc[manifesto["arquivo"] == arquivo.name]
    hash_registrado = None if registro_manifesto.empty else registro_manifesto.iloc[0]["sha256"]
    integridade = hash_registrado == calcular_sha256(arquivo)

    lat_grade = float(dados["lat_grade_nsrdb"].iloc[0])
    lon_grade = float(dados["lon_grade_nsrdb"].iloc[0])
    distancia_grade = distancia_haversine_km(
        local["lat"], local["lon"], lat_grade, lon_grade
    )

    datas = pd.DatetimeIndex(dados["data"])
    resumo_dados.append(
        {
            "Localidade": local["nome"],
            "Registros": len(dados),
            "Data inicial": datas.min().date(),
            "Data final": datas.max().date(),
            "Dias ausentes": len(datas_esperadas.difference(datas)),
            "Datas duplicadas": int(datas.duplicated().sum()),
            "Distância até a grade (km)": distancia_grade,
            "GHI mínimo": dados["ghi"].min(),
            "GHI médio": dados["ghi"].mean(),
            "GHI máximo": dados["ghi"].max(),
            "Unidade": dados["ghi_unidade_api"].iloc[0],
            "Consistência": "Adequada" if valido and integridade else motivo,
        }
    )

df_resumo_dados = pd.DataFrame(resumo_dados)
df_resumo_dados["Distância até a grade (km)"] = df_resumo_dados[
    "Distância até a grade (km)"
].round(2)
df_resumo_dados[["GHI mínimo", "GHI médio", "GHI máximo"]] = df_resumo_dados[
    ["GHI mínimo", "GHI médio", "GHI máximo"]
].round(2)

display(df_resumo_dados)

Os dez arquivos apresentam 2.192 observações diárias, sem lacunas ou duplicações no período analisado. As distâncias entre as fábricas e os pontos da grade permanecem compatíveis com a resolução espacial do produto utilizado.

## 3. Análise exploratória das séries de GHI

### 3.1 Comparação entre as localidades

Para facilitar a visualização de longo prazo, os dados diários foram convertidos em médias mensais. Essa transformação foi utilizada exclusivamente nos gráficos exploratórios; o treinamento dos modelos permaneceu baseado nas séries diárias.

In [ ]:
series_mensais = []
for nome, dados in bases.items():
    mensal = (
        dados.set_index("data")[["ghi"]]
        .resample("ME")
        .mean()
        .reset_index()
    )
    mensal["localidade"] = nome
    series_mensais.append(mensal)

ghi_mensal = pd.concat(series_mensais, ignore_index=True)

fig, ax = plt.subplots(figsize=(15, 7))
for nome, grupo in ghi_mensal.groupby("localidade"):
    ax.plot(grupo["data"], grupo["ghi"], linewidth=1.5, alpha=0.9, label=nome)

ax.set_title("GHI mensal médio nas dez localidades, 2019–2024")
ax.set_xlabel("Data")
ax.set_ylabel("GHI médio diário [W/m²]")
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Séries históricas por localidade

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(15, 18), sharex=True, sharey=True)

for ax, local in zip(axes.flat, LOCALIDADES_EV):
    grupo = ghi_mensal.loc[ghi_mensal["localidade"] == local["nome"]]
    ax.plot(grupo["data"], grupo["ghi"], color="#F9A825", linewidth=1.5)
    ax.set_title(local["nome"], fontsize=10)
    ax.set_ylabel("GHI [W/m²]")
    ax.grid(True, alpha=0.3)

fig.suptitle("Séries mensais de GHI por localidade", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### 3.3 Médias anuais de GHI

In [ ]:
ghi_anual = (
    pd.concat(bases.values(), ignore_index=True)
    .groupby(["localidade", "ano"], as_index=False)["ghi"]
    .mean()
    .pivot(index="localidade", columns="ano", values="ghi")
)

fig, ax = plt.subplots(figsize=(10, 6))
imagem = ax.imshow(ghi_anual.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(ghi_anual.columns)), ghi_anual.columns)
ax.set_yticks(range(len(ghi_anual.index)), ghi_anual.index)
ax.set_title("GHI médio anual por localidade [W/m²]")

for i in range(ghi_anual.shape[0]):
    for j in range(ghi_anual.shape[1]):
        ax.text(j, i, f"{ghi_anual.iloc[i, j]:.1f}", ha="center", va="center", fontsize=8)

fig.colorbar(imagem, ax=ax, label="GHI [W/m²]")
plt.tight_layout()
plt.show()

A comparação evidencia diferenças sazonais e níveis médios distintos entre as localidades. As maiores médias do período foram observadas em San Luis Potosí, Casa Grande e Camaçari, enquanto os menores valores médios ocorreram nas localidades da região de Detroit e Dearborn.

## 4. Aplicação do pipeline de previsão

O processamento aplicado a cada localidade seguiu a metodologia descrita no notebook teórico do projeto. As mesmas etapas e os mesmos hiperparâmetros foram utilizados em todas as séries, permitindo a comparação dos resultados em condições equivalentes.

| Etapa | Procedimento adotado |
|---|---|
| Preparação da série | Ordenação cronológica, remoção de valores inválidos e agregação diária |
| Quantização | Conversão do GHI em 128 níveis discretos |
| Normalização | Escalonamento Min-Max para o intervalo entre 0 e 1 |
| Variáveis temporais | Defasagens de 1, 2, 3 e 7 dias |
| Médias móveis | Janelas de 3, 7 e 30 dias, calculadas somente com valores anteriores |
| Variável-alvo | GHI normalizado do dia seguinte |
| Divisão dos dados | 80% para treinamento e 20% para teste, em ordem cronológica |
| Modelos | XGBoost e MLPRegressor |
| Métricas | MAE, MSE, RMSE e coeficiente de determinação R² |

O XGBoost foi configurado com 300 estimadores, profundidade máxima igual a 3 e taxa de aprendizagem de 0,05. A rede neural MLP foi composta por duas camadas ocultas, com 64 e 32 neurônios, função de ativação ReLU e otimizador Adam.

Os parâmetros de quantização e normalização foram ajustados exclusivamente com o conjunto de treinamento, evitando a utilização de informações futuras durante a preparação dos dados.

## 5. Resultados dos modelos

As métricas apresentadas nesta seção foram calculadas sobre a parcela final de 20% de cada série. Como a variável-alvo foi quantizada e normalizada, os valores de MAE, MSE e RMSE são adimensionais e se referem à escala normalizada entre 0 e 1.

In [ ]:
metricas_file = RESULTADOS_DIR / "metricas_geral.csv"
resumo_file = RESULTADOS_DIR / "resumo_localidades.csv"

if not metricas_file.exists() or not resumo_file.exists():
    raise FileNotFoundError("Os resultados dos modelos não foram localizados.")

df_metricas = pd.read_csv(metricas_file)
df_resumo_modelos = pd.read_csv(resumo_file)

metricas_exibicao = df_metricas.copy()
metricas_exibicao[["MAE", "MSE", "RMSE", "R2"]] = metricas_exibicao[
    ["MAE", "MSE", "RMSE", "R2"]
].round(4)
metricas_exibicao = metricas_exibicao.rename(columns={"R2": "R²"})

display(metricas_exibicao)

### 5.1 Comparação das métricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

configuracoes = [
    ("R2", "Coeficiente de determinação R²", "R²"),
    ("RMSE", "Raiz do erro quadrático médio", "RMSE"),
    ("MAE", "Erro absoluto médio", "MAE"),
]

for ax, (coluna, titulo, rotulo_y) in zip(axes, configuracoes):
    tabela = df_metricas.pivot(index="Localidade", columns="Modelo", values=coluna)
    tabela.plot(kind="bar", ax=ax, color=["#1565C0", "#F9A825"])
    ax.set_title(titulo)
    ax.set_xlabel("")
    ax.set_ylabel(rotulo_y)
    ax.tick_params(axis="x", rotation=75)
    ax.legend(title="Modelo")

plt.tight_layout()
plt.show()

### 5.2 Comparação entre valores observados e previstos

Os gráficos a seguir apresentam os valores reais e previstos no conjunto de teste. Essa representação permite observar o comportamento temporal dos dois modelos em cada localidade.

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(16, 20), sharex=False, sharey=True)

for ax, local in zip(axes.flat, LOCALIDADES_EV):
    arquivo = PREVISOES_DIR / nome_arquivo(local["nome"]) / "previsoes_modelos.csv"
    previsoes = pd.read_csv(arquivo, parse_dates=["data"])

    ax.plot(previsoes["data"], previsoes["ghi_real"], color="#212121", linewidth=1.4, label="Observado")
    ax.plot(previsoes["data"], previsoes["ghi_previsto_xgboost"], color="#1565C0", linewidth=1.0, alpha=0.85, label="XGBoost")
    ax.plot(previsoes["data"], previsoes["ghi_previsto_mlp"], color="#F9A825", linewidth=1.0, alpha=0.85, label="MLP")
    ax.set_title(local["nome"], fontsize=10)
    ax.set_ylabel("GHI normalizado")
    ax.grid(True, alpha=0.25)

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.01))
fig.suptitle("Valores observados e previstos no conjunto de teste", fontsize=16, y=1.025)
plt.tight_layout()
plt.show()

### 5.3 Síntese por localidade

In [ ]:
resumo_exibicao = df_resumo_modelos[
    [
        "Localidade", "XGBoost_MAE", "XGBoost_RMSE", "XGBoost_R2",
        "MLP_MAE", "MLP_RMSE", "MLP_R2", "Melhor_Modelo",
    ]
].copy()

colunas_numericas = [
    "XGBoost_MAE", "XGBoost_RMSE", "XGBoost_R2",
    "MLP_MAE", "MLP_RMSE", "MLP_R2",
]
resumo_exibicao[colunas_numericas] = resumo_exibicao[colunas_numericas].round(4)
resumo_exibicao = resumo_exibicao.rename(
    columns={
        "XGBoost_R2": "XGBoost R²",
        "MLP_R2": "MLP R²",
        "Melhor_Modelo": "Modelo com maior R²",
    }
)

display(resumo_exibicao)

## 6. Discussão dos resultados

In [ ]:
melhor_xgb = df_resumo_modelos.loc[df_resumo_modelos["XGBoost_R2"].idxmax()]
melhor_mlp = df_resumo_modelos.loc[df_resumo_modelos["MLP_R2"].idxmax()]
vitorias_xgb = int((df_resumo_modelos["XGBoost_R2"] > df_resumo_modelos["MLP_R2"]).sum())
vitorias_mlp = int((df_resumo_modelos["MLP_R2"] > df_resumo_modelos["XGBoost_R2"]).sum())

texto_discussao = f"""
Os resultados indicam que o desempenho dos modelos varia de acordo com a localidade e com a regularidade da série solar. O maior coeficiente de determinação do XGBoost foi obtido para **{melhor_xgb['Localidade']}**, com R² igual a **{melhor_xgb['XGBoost_R2']:.4f}**. A MLP também apresentou seu maior R² nessa localidade, com valor igual a **{melhor_mlp['MLP_R2']:.4f}**.

Considerando o maior R² em cada localidade, o XGBoost apresentou melhor resultado em **{vitorias_xgb} localidades**, enquanto a MLP apresentou melhor resultado em **{vitorias_mlp} localidades**. Essa distribuição mostra que nenhum dos modelos foi superior em todos os casos.

Os melhores desempenhos foram observados em Fremont, Nevada e Casa Grande. Nessas localidades, os modelos reproduziram de maneira mais consistente a variação temporal do GHI. Em Camaçari, Texas, Georgia e San Luis Potosí, os menores valores de R² indicam maior dificuldade na representação das oscilações diárias por meio apenas das defasagens e médias móveis utilizadas.

De forma geral, os resultados demonstram que o pipeline é aplicável às diferentes séries, mas também evidenciam que a capacidade preditiva depende das características climáticas de cada região. A inclusão futura de variáveis meteorológicas adicionais, como nebulosidade, temperatura, umidade e velocidade do vento, poderá ampliar a capacidade explicativa dos modelos.
"""

display(Markdown(texto_discussao))

## 7. Conclusão

O estudo reuniu séries diárias de GHI para dez localidades associadas a fábricas de veículos elétricos e aplicou, de forma padronizada, um pipeline de previsão baseado em XGBoost e MLP.

A base utilizada apresentou cobertura diária completa entre 2019 e 2024, com origem e coordenadas documentadas. A análise exploratória permitiu identificar diferenças sazonais e níveis médios de irradiância distintos entre as regiões estudadas.

Na etapa de modelagem, os dois algoritmos apresentaram resultados competitivos, sem predominância absoluta de um único método. O melhor desempenho geral foi obtido para a Tesla Fremont Factory. Os resultados confirmam a viabilidade do procedimento adotado e estabelecem uma referência para etapas posteriores de aprimoramento do modelo e inclusão de novas variáveis explicativas.

## Referências e fontes de dados

- National Laboratory of the Rockies. **NSRDB GOES Aggregated PSM v4**. Disponível em: <https://developer.nlr.gov/docs/solar/nsrdb/nsrdb-GOES-aggregated-v4-0-0-download/>.
- OpenStreetMap Contributors. **OpenStreetMap**. Disponível em: <https://www.openstreetmap.org/>.
- Nominatim. **Open-source geocoding with OpenStreetMap data**. Disponível em: <https://nominatim.org/>.
- As páginas institucionais utilizadas para identificação das fábricas encontram-se relacionadas na Tabela da Seção 2.2.
- XGBoost Documentation. Disponível em: <https://xgboost.readthedocs.io/>.
- Scikit-learn. **MLPRegressor**. Disponível em: <https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html>.